In [8]:
import pyscf
import numpy as np

mol = pyscf.gto.Mole()
mol.atom = '''
O 0.0 0.0 0.0
H 0.0 1.0 0.0
H 0.0 0.0 1.0
'''
mol.basis = 'sto-3g'
mol.build()

mdft = pyscf.scf.RKS(mol)
mdft.xc = 'b3lyp'
mdft.kernel()

dm1 = mdft.make_rdm1(ao_repr=True)
ao_value = pyscf.dft.numint.eval_ao(mol, mdft.grids.coords, deriv=2)
rho_dft = pyscf.dft.numint.eval_rho(mol, ao_value, dm1, xctype="GGA")

lda_grids = pyscf.dft.libxc.eval_xc("LDA,", rho_dft[0], 0)[0]
print(np.linalg.norm((lda_grids / (-3 / 4 * (3 / np.pi) ** (1 / 3))) ** 3 - rho_dft[0]))

converged SCF energy = -75.3186721647241
2.391172746150172e-12


In [7]:
import pyscf
import numpy as np

mol = pyscf.gto.Mole()
mol.atom = """
O 0.0 0.0 0.0
H 1.0 0.0 0.0
H 0.0 1.0 0.0
H 0.0 0.0 1.0
"""
mol.basis = "sto-3g"
mol.spin = 1
mol.build()

mdft = pyscf.scf.UKS(mol)
mdft.xc = "lda,"
mdft.kernel()

dm1 = mdft.make_rdm1(ao_repr=True)
ao_value = pyscf.dft.numint.eval_ao(mol, mdft.grids.coords, deriv=0)
rho_dfta = pyscf.dft.numint.eval_rho(mol, ao_value, dm1[0], xctype="LDA")
rho_dftb = pyscf.dft.numint.eval_rho(mol, ao_value, dm1[1], xctype="LDA")

lda_grids = pyscf.dft.libxc.eval_xc("LDA,", [rho_dfta, rho_dftb], 1)[0]
lda_grids1 = pyscf.dft.libxc.eval_xc("LDA,", rho_dfta + rho_dftb, 0)[0]

h1e = mol.intor("int1e_nuc") + mol.intor("int1e_kin")
ene_h1e = np.einsum("pq,pq", h1e, dm1[0] + dm1[1])
eri = mol.intor("int2e")
ene_eris = 0.5 * np.einsum("pqrs,pq,rs", eri, dm1[0] + dm1[1], dm1[0] + dm1[1])

print(mdft.e_tot - (ene_h1e + ene_eris + mdft.energy_nuc()))
print(np.sum(lda_grids * (rho_dfta + rho_dftb) * mdft.grids.weights))
print(np.sum(lda_grids1 * (rho_dfta + rho_dftb) * mdft.grids.weights))


# print(np.linalg.norm(lda_grids * mdft.grids.weights - lda_grids1 * mdft.grids.weights))
# print(
#     np.linalg.norm(
#         (lda_grids / (-3 / 4 * (3 / np.pi) ** (1 / 3))) ** 3 - (rho_dfta + rho_dftb)
#     )
# )
# print(
#     np.linalg.norm(
#         (lda_grids1 / (-3 / 4 * (3 / np.pi) ** (1 / 3))) ** 3 - (rho_dfta + rho_dftb)
#     )
# )

converged SCF energy = -74.3464802782786  <S^2> = 0.75094459  2S+1 = 2.0009444
-8.514815598900114
-8.514815598900054
-8.495567508459256
-0.019248090440796554


In [14]:
rho_dft[0]

array([7.64091893e-11, 6.54553103e-12, 5.73878593e-09, ...,
       9.80549478e+01, 9.80549478e+01, 9.80549478e+01])